# 서브 워드 비교 실험

서브워드 라이브러리 HuggingFace Tokenizer와 SentencePiece를 비교하는 과제 노트북입니다.

진행 순서는 다음과 같습니다.

1. 데이터 준비 : 네이버 영화 리뷰 [[링크]](https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt)
2. Sentencepiece/HuggingFace를 사용하여 단어사전 구축 및 속도 측정
3. 1과 2 결과물의 상위 빈도 30개의 서브워드 분석
4. 학습 속도(단어사전 구축 속도), 단일 문장 변환 속도, 코퍼스 변환 속도 차이 특정
5. 기타 추가 실험
6. 사용성과 장단점 분석

## 1. 데이터 준비 : 네이버 영화 리뷰

### HuggingFace Tokenizer와 SentencePiece 설치하기

In [1]:
!pip install -q sentencepiece tokenizers

### 네이버 리뷰 데이터 불러오기

In [6]:
import requests
from pathlib import Path

url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
corpus_path = Path("data/raw/corpus.txt")
corpus_path.parent.mkdir(parents=True, exist_ok=True)

if not corpus_path.exists():
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    corpus_path.write_bytes(response.content)

print(f"saved: {corpus_path.resolve()}")

saved: C:\Users\82105\Desktop\nlp\corpus.txt


In [7]:
import os

file_path = "data/raw/corpus.txt"

if os.path.exists(file_path):
    print("다운로드 완료")
else:
    print("파일을 찾을 수 없음")

다운로드 완료


### 데이터 정제하기
현재 데이터에는 `id`, `document`, `label`이 있습니다. 이 중 실질적인 문장 데이터인 `document`만 활용합니다.

In [22]:
import csv
from pathlib import Path

corpus_path = Path("data/raw/corpus.txt")
corpus_path.parent.mkdir(parents=True, exist_ok=True)
clean_corpus_path = Path("data/processed/corpus_documents.txt")
clean_corpus_path.parent.mkdir(parents=True, exist_ok=True)
raw_row_count = 0
doc_count = 0
sample_sentence = None

with corpus_path.open("r", encoding="utf-8") as src, clean_corpus_path.open("w", encoding="utf-8", newline="") as dst:
    reader = csv.DictReader(src, delimiter="\t")
    for row in reader:
        raw_row_count += 1
        document = (row.get("document") or "").strip()
        if not document:
            continue
        if sample_sentence is None:
            sample_sentence = document
        dst.write(document + "\n")
        doc_count += 1

print(f"raw rows: {raw_row_count:,}")
print(f"정제 후 문서 수: {doc_count:,}")
print(f"sample: {sample_sentence}")

raw rows: 150,000
정제 후 문서 수: 149,995
sample: 아 더빙.. 진짜 짜증나네요 목소리


### 2. Sentencepiece/HuggingFace를 사용하여 단어사전 구축 및 속도 측정

HuggingFace BPE는 SentencePiece와 내제된 철학이 다릅니다.  
고전적인 BPE는 단어 단위 알고리즘에서 출발하였으며 단어 목록을 입력으로 받아 문장 분리 -> 단어 분리 -> BPE를 가정합니다.  
때문에 이 철학을 따르는 HuggingFace BPE는 `Witespace()` 혹은 `ByteLevel()`과 같은 pre-tokenizer가 필요합니다.  
반면 SentencePiece는 공백도 학습하자는 철학으로 공백에 대하여 \_\_로 처리하여 기본 문장에서 공백을 기준으로 나누는 것은 동일하지만 문장 전체를 입력받게 되며 다음과 같이 처리합니다.  

"오늘은 날이 참 밝다."  
"오늘은\_\_날이\_\_참\_\_밝다."

만일 SentencePiece의 철학과 유사하게 진행하려면 `Metqaspace()`를 활용하십시오.

#### Sentencepiece를 사용하여 단어사전 구축

In [23]:
import time
import sentencepiece as spm
from pathlib import Path

vocab_size = 8000
sp_prefix = "models/sentencepiece/spm_nsmc_bpe"
Path(sp_prefix).parent.mkdir(parents=True, exist_ok=True)

start = time.perf_counter()
spm.SentencePieceTrainer.Train(
    input=str(clean_corpus_path),
    model_prefix=sp_prefix,
    vocab_size=vocab_size,
    model_type="bpe",
    character_coverage=0.9995,
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    num_threads=4,
)
sp_train_time = time.perf_counter() - start

sp = spm.SentencePieceProcessor(model_file=f"{sp_prefix}.model")
print(f"SentencePiece vocab size={sp.get_piece_size():,}")
print(f"SentencePiece training time: {sp_train_time:.3f} sec")

SentencePiece vocab size=8,000
SentencePiece training time: 22.253 sec


#### HuggingFace Tokenizer를 사용하여 단어사전 구축

In [11]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.normalizers import NFKC
from tokenizers.pre_tokenizers import Metaspace
from tokenizers.decoders import Metaspace as MetaspaceDecoder
from tokenizers.trainers import BpeTrainer
from pathlib import Path

hf_tokenizer = Tokenizer(BPE(unk_token="<unk>"))
hf_tokenizer.normalizer = NFKC()

try:
    hf_tokenizer.pre_tokenizer = Metaspace(replacement="▁", prepend_scheme="always")
except TypeError:
    hf_tokenizer.pre_tokenizer = Metaspace(replacement="▁", add_prefix_space=True)

hf_tokenizer.decoder = MetaspaceDecoder(replacement="▁")

hf_trainer = BpeTrainer(
    vocab_size=vocab_size,
    special_tokens=["<pad>", "<unk>", "<s>", "</s>"],
    show_progress=True,
)

start = time.perf_counter()
hf_tokenizer.train([str(clean_corpus_path)], trainer=hf_trainer)
hf_train_time = time.perf_counter() - start
Path("models/huggingface").mkdir(parents=True, exist_ok=True)
hf_tokenizer.save("models/huggingface/hf_bpe_tokenizer.json")

print(f"HuggingFace vocab size: {hf_tokenizer.get_vocab_size():,}")
print(f"HuggingFace training time: {hf_train_time:.3f} sec")

HuggingFace vocab size: 8,000
HuggingFace training time: 4.738 sec


### 3. 1과 2 결과물의 상위 빈도 30개의 서브워드 분석

#### 상위 30개 기준, 단어사전 조회하기

In [ ]:
import sys
!{sys.executable} -m pip install pandas

In [17]:
import pandas as pd

documents = clean_corpus_path.read_text(encoding="utf-8").splitlines()
print(f"loaded documents: {len(documents):,}")

sp_vocab_top30 = [(i, sp.id_to_piece(i)) for i in range(30)]
hf_vocab_top30 = [(i, hf_tokenizer.id_to_token(i)) for i in range(30)]

vocab_compare = pd.DataFrame({
    "rank/id": range(30),
    "sentencepiece": [piece for _, piece in sp_vocab_top30],
    "huggingface": [token for _, token in hf_vocab_top30],
})
vocab_compare

loaded documents: 149,995


,rank/id,sentencepiece,huggingface
0,0,<pad>,<pad>
1,1,<unk>,<unk>
2,2,<s>,<s>
3,3,</s>,</s>
4,4,..,\n
5,5,영화,!
6,6,▁영화,""""
7,7,▁이,#
8,8,▁아,$
9,9,...,%


### 분절된 토큰 빈도 조회

In [18]:
from collections import Counter

def count_sp_tokens(texts):
    counter = Counter()
    for text in texts:
        counter.update(sp.encode(text, out_type=str))
    return counter

def count_hf_tokens(texts):
    counter = Counter()
    for text in texts:
        counter.update(hf_tokenizer.encode(text).tokens)
    return counter

sp_token_freq = count_sp_tokens(documents)
hf_token_freq = count_hf_tokens(documents)

freq_compare = pd.DataFrame({
    "sp_token": [token for token, _ in sp_token_freq.most_common(30)],
    "sp_count": [count for _, count in sp_token_freq.most_common(30)],
    "hf_token": [token for token, _ in hf_token_freq.most_common(30)],
    "hf_count": [count for _, count in hf_token_freq.most_common(30)],
})
freq_compare

,sp_token,sp_count,hf_token,hf_count
0,.,67420,▁,22387
1,..,28463,이,21372
2,▁영화,22680,.,18405
3,...,22548,의,17526
4,이,17893,▁영화,16450
5,",",17371,도,16158
6,▁,16297,..,15841
7,의,15251,에,15270
8,도,13850,...,15218
9,을,13203,을,15200


### 4. 학습 속도(단어사전 구축 속도), 단일 문장 변환 속도, 코퍼스 변환 속도 차이 특정

#### 단일 문장 변환 속도 측정

In [19]:
single_sentence = "이 영화는 배우들의 연기가 좋았지만 결말은 조금 아쉬웠다."
repeat = 5000

start = time.perf_counter()
for _ in range(repeat):
    sp_single_tokens = sp.encode(single_sentence, out_type=str)
sp_single_time = time.perf_counter() - start

start = time.perf_counter()
for _ in range(repeat):
    hf_single_tokens = hf_tokenizer.encode(single_sentence).tokens
hf_single_time = time.perf_counter() - start

single_speed = pd.DataFrame([
    {"tokenizer": "SentencePiece", "total_sec": sp_single_time, "ms_per_encode": sp_single_time / repeat * 1000, "tokens": sp_single_tokens},
    {"tokenizer": "HuggingFace", "total_sec": hf_single_time, "ms_per_encode": hf_single_time / repeat * 1000, "tokens": hf_single_tokens},
])
single_speed

,tokenizer,total_sec,ms_per_encode,tokens
0,SentencePiece,0.055804,0.011161,"[▁이, ▁영화는, ▁배우들의, ▁연기가, ▁좋았지만, ▁결말은, ▁조금, ▁아쉬,..."
1,HuggingFace,0.112022,0.022404,"[▁이, ▁영화는, ▁배우들의, ▁연기가, ▁좋았지만, ▁결말은, ▁조금, ▁아쉬웠..."


#### 단일 문장 변환 속도 결과 해석

- 저장된 실행 결과 기준으로 SentencePiece는 약 `0.011 ms/문장`, HuggingFace Tokenizer는 약 `0.022 ms/문장`이 걸렸다. 단일 문장 반복 변환에서는 SentencePiece가 더 빠르게 측정되었다.
- 단일 문장 측정은 실제 토큰화 연산뿐 아니라 Python에서 함수를 반복 호출하는 오버헤드의 영향을 크게 받는다. 따라서 이 결과는 "짧은 입력을 한 문장씩 처리할 때의 체감 속도"에 가깝다.
- 두 토크나이저의 분절 결과는 대체로 비슷하지만 완전히 같지는 않다. 예를 들어 같은 문장에서도 HuggingFace BPE는 일부 어절을 더 잘게 나눌 수 있고, SentencePiece는 학습된 piece에 따라 더 긴 단위로 묶을 수 있다.
- 결론적으로 실시간으로 짧은 문장을 하나씩 처리하는 상황에서는 이 실험 기준 SentencePiece가 유리하게 나타났다. 다만 반복 횟수, 문장 길이, CPU 상태에 따라 작은 차이는 바뀔 수 있다.

### 코퍼스 전체 변환 속도 측정

In [20]:
start = time.perf_counter()
sp_corpus_tokens = [sp.encode(text, out_type=str) for text in documents]
sp_corpus_time = time.perf_counter() - start

start = time.perf_counter()
hf_corpus_tokens = [encoding.tokens for encoding in hf_tokenizer.encode_batch(documents)]
hf_corpus_time = time.perf_counter() - start

corpus_speed = pd.DataFrame([
    {"tokenizer": "SentencePiece", "total_sec": sp_corpus_time, "docs_per_sec": len(documents) / sp_corpus_time, "total_tokens": sum(map(len, sp_corpus_tokens))},
    {"tokenizer": "HuggingFace", "total_sec": hf_corpus_time, "docs_per_sec": len(documents) / hf_corpus_time, "total_tokens": sum(map(len, hf_corpus_tokens))},
])
corpus_speed

,tokenizer,total_sec,docs_per_sec,total_tokens
0,SentencePiece,4.693009,31961.369264,2494019
1,HuggingFace,2.905544,51623.728805,2555709


#### 코퍼스 전체 변환 속도 결과 해석

- 저장된 실행 결과 기준으로 SentencePiece는 전체 코퍼스 변환에 약 `4.69초`, HuggingFace Tokenizer는 약 `2.91초`가 걸렸다. 문서 처리량도 SentencePiece는 약 `31,961 docs/sec`, HuggingFace는 약 `51,624 docs/sec`로 HuggingFace가 더 높았다.
- 단일 문장 변환과 달리 전체 코퍼스 변환에서는 HuggingFace의 `encode_batch()`가 효과적으로 작동한다. Rust 기반 구현과 배치 처리 덕분에 많은 문장을 한꺼번에 변환할 때 속도 이점이 커진다.
- 전체 토큰 수는 SentencePiece가 약 `2,494,019개`, HuggingFace가 약 `2,555,709개`로 HuggingFace 쪽이 조금 더 많았다. 이는 HuggingFace BPE가 같은 코퍼스를 약간 더 잘게 분절했다는 의미다.
- 결론적으로 단일 문장 처리에서는 SentencePiece가 빠르게 나왔지만, 대량 코퍼스 처리에서는 HuggingFace Tokenizer가 더 적합한 결과를 보였다. 실제 모델 학습 전 대규모 데이터셋을 미리 토큰화하는 상황이라면 HuggingFace의 배치 API가 장점이 된다.

## 5. 기타 추가 실험

In [21]:
from IPython.display import display

def token_stats(name, tokenized):
    token_lengths = [len(tokens) for tokens in tokenized]
    total_chars = sum(len(text) for text in documents)
    total_tokens = sum(token_lengths)
    flat_tokens = [token for tokens in tokenized for token in tokens]
    return {
        "tokenizer": name,
        "avg_tokens_per_doc": sum(token_lengths) / len(token_lengths),
        "chars_per_token": total_chars / total_tokens,
        "unique_tokens_used": len(set(flat_tokens)),
        "unk_count": flat_tokens.count("<unk>"),
    }

extra_stats = pd.DataFrame([
    token_stats("SentencePiece", sp_corpus_tokens),
    token_stats("HuggingFace", hf_corpus_tokens),
])

test_sentences = [
    "완전 꿀잼ㅋㅋ 배우들 연기 미쳤다!",
    "스토리는 별로였지만 OST는 계속 생각난다.",
    "This movie was surprisingly good :)",
]

examples = pd.DataFrame({
    "sentence": test_sentences,
    "sentencepiece": [sp.encode(text, out_type=str) for text in test_sentences],
    "huggingface": [hf_tokenizer.encode(text).tokens for text in test_sentences],
})

display(extra_stats)
display(examples)

,tokenizer,avg_tokens_per_doc,chars_per_token,unique_tokens_used,unk_count
0,SentencePiece,16.627348,2.117267,9188,0
1,HuggingFace,17.038628,2.066160,7681,0


,sentence,sentencepiece,huggingface
0,완전 꿀잼ㅋㅋ 배우들 연기 미쳤다!,"[▁완전, ▁꿀잼, ᄏᄏ, ▁배우들, ▁연기, ▁미, 쳤다, !]","[▁완전, ▁꿀잼, ᄏᄏ, ▁배우들, ▁연기, ▁미쳤, 다!]"
1,스토리는 별로였지만 OST는 계속 생각난다.,"[▁스토리는, ▁별로, 였지만, ▁OST, 는, ▁계속, ▁생각난다, .]","[▁스토리는, ▁별로였, 지만, ▁OST, 는, ▁계속, ▁생각, 난다.]"
2,This movie was surprisingly good :),"[▁T, h, is, ▁m, ovie, ▁w, as, ▁s, u, r, p, r, ...","[▁T, h, i, s, ▁m, ov, ie, ▁w, a, s, ▁s, u, r, ..."


### 기타 추가 실험 결과 해석

- `avg_tokens_per_doc`는 문서 하나가 평균 몇 개의 서브워드로 나뉘는지를 나타낸다. 저장된 결과 기준 SentencePiece는 약 `16.63개`, HuggingFace는 약 `17.04개`로 HuggingFace가 조금 더 많은 토큰을 만들었다. 이는 HuggingFace가 평균적으로 더 세밀하게 분절했다는 뜻이다.
- `chars_per_token`은 토큰 하나가 평균 몇 글자를 담당하는지를 보여준다. SentencePiece가 약 `2.12`, HuggingFace가 약 `2.07`로 SentencePiece 토큰이 약간 더 긴 단위를 포착했다.
- `unique_tokens_used`는 실제 코퍼스 변환 과정에서 사용된 서로 다른 토큰의 수다. SentencePiece 쪽이 더 넓은 종류의 토큰을 사용했고, HuggingFace는 상대적으로 더 제한된 vocab 범위 안에서 반복적으로 토큰을 사용했다.
- `unk_count`가 두 방식 모두 `0`으로 나타난 것은 이 코퍼스와 예문 범위에서는 명시적인 `<unk>` 토큰이 거의 필요하지 않았다는 뜻이다. 서브워드 방식이 희귀 표현을 더 작은 단위로 쪼개 처리했기 때문이다.
- 예문 비교에서 `꿀잼ㅋㅋ`, `OST`, 영어 문장처럼 학습 데이터에서 상대적으로 덜 일반적인 표현은 두 토크나이저의 차이가 더 잘 드러난다. SentencePiece는 공백 표시 `▁`를 유지하면서 문장 전체 흐름을 반영하고, HuggingFace는 Metaspace pre-tokenizer와 BPE 병합 결과에 따라 일부 단어를 더 짧은 조각으로 나눈다.
- 종합하면 SentencePiece는 평균적으로 조금 더 긴 토큰을 만들고 단일 문장 처리에서 빠르게 측정되었으며, HuggingFace Tokenizer는 배치 변환 속도가 빠르고 대량 코퍼스 처리에 강점을 보였다.

## 6. 사용성과 장단점 분석

**장점**  

- SentencePiece는 원문 문장을 그대로 입력해도 공백을 `▁` 기호로 포함해 학습하므로 한국어처럼 명확한 띄어쓰기 기반 토큰화가 항상 안정적이지 않은 언어에서 사용하기 쉽다.
- SentencePiece는 모델 파일 하나로 학습 결과를 저장하고 재사용하기 쉬우며, 별도 pre-tokenizer를 크게 신경 쓰지 않아도 된다.
- HuggingFace Tokenizers는 Rust 기반 구현이라 코퍼스 일괄 변환에서 빠른 편이고, normalizer/pre-tokenizer/trainer/decoder를 조합해 실험 조건을 세밀하게 통제할 수 있다.
- 두 방식 모두 희귀어와 신조어를 문자 또는 짧은 서브워드 조합으로 처리할 수 있어 단어 단위 토큰화보다 OOV 문제가 작다.

**단점**  

- SentencePiece는 학습 옵션이 간단한 대신 HuggingFace Tokenizers처럼 전처리 파이프라인을 세밀하게 조합하기는 어렵다.
- HuggingFace BPE는 pre-tokenizer 선택에 따라 결과가 크게 달라진다. Whitespace를 쓰면 SentencePiece와 철학이 달라지고, Metaspace를 써야 공백을 포함하는 비교가 비교적 공정해진다.
- 두 토크나이저의 상위 vocab id는 실제 코퍼스 빈도 순위와 같지 않다. 실제 사용 빈도 분석은 학습된 토크나이저로 코퍼스를 다시 분절한 뒤 Counter로 세는 방식이 필요하다.

**실험 해석**  

- 학습 속도는 실행 환경과 라이브러리 버전에 따라 달라지므로 같은 머신에서 여러 번 반복 측정하는 것이 좋다.
- 단일 문장 변환은 호출 오버헤드의 영향을 많이 받고, 전체 코퍼스 변환은 배치 API 지원 여부의 영향을 크게 받는다.
- 한국어 리뷰 데이터에서는 조사, 어미, 감탄 표현, 반복 문자(`ㅋㅋ`, `ㅎㅎ`)가 상위 빈도 서브워드에 많이 나타나는지 확인하면 두 토크나이저의 분절 차이를 해석하기 쉽다.